# 06. 벡터 RAG (Vector RAG) 구현

앞 장들에서 **임베딩(Embedding)**, **벡터DB(VectorDB)**, **LLM** 을 각각 따로 배웠습니다.
이번 장에서는 이 부품들을 하나로 조립해서, 논문 2편에 대해 질문하면 **논문 내용을 근거로 답하는 RAG** 를 완성합니다.

실험에 비유하면 이렇습니다.

- **LLM 혼자** = 논문을 읽지 않고 기억에만 의존해 답하는 연구원 → 그럴듯하지만 틀린 말(환각, Hallucination)을 할 수 있음
- **RAG** = 질문을 받으면 먼저 **실험노트·논문 보관함에서 관련 페이지를 찾아 펼쳐 놓고**, 그 내용을 보면서 답하는 연구원

## 학습 목표

1. RAG 의 8단계 파이프라인(문서 로드 → 분할 → 임베딩 → 저장 → 검색 → 프롬프트 → LLM → 체인)을 이해합니다.
2. **Part A**: numpy 와 순수 파이썬만으로 RAG 를 직접 만들어 봅니다(`naive_rag` 함수 하나).
3. **Part B**: 같은 RAG 를 LangChain 으로 다시 만들고, "라이브러리 = 방금 만든 것을 편하게 포장한 것" 임을 확인합니다.
4. **Part C**: 질문·조각 크기·검색 개수를 바꿔 보며 벡터 RAG 가 **잘하는 것과 못하는 것**을 직접 확인합니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # 불필요한 경고 메시지 숨기기

from pathlib import Path
import numpy as np

# 공용 모듈: 모델 설정, 논문 로드/분할 함수
from skku_rag import get_llm, get_embeddings, load_papers, split_documents, LLM_MODEL, EMBED_MODEL

print("LLM 모델     :", LLM_MODEL)
print("임베딩 모델  :", EMBED_MODEL)

## RAG 파이프라인 한눈에 보기

RAG 는 크게 **사전작업(Pre-processing)** 과 **실행(Runtime)** 두 부분으로 나뉩니다.
사전작업은 "시료를 정리해 보관함에 넣어 두는 일"로 **한 번만** 하면 되고,
실행은 "질문이 들어올 때마다 보관함에서 꺼내 답하는 일"로 **질문마다** 반복합니다.

```
 [사전작업: 한 번만]  ───────────────────────────────────────────────────────────
   논문 PDF ──► ① 문서 로드 ──► ② 분할(Chunk) ──► ③ 임베딩 ──► ④ 벡터DB 저장
               (페이지 읽기)   (작은 조각으로)   (숫자 벡터로)   (보관함에 정리)
                                                                     │
 [실행: 질문마다]  ──────────────────────────────────────────────────┼──────────
                                                                     ▼
   질문 ──────────────────────────────────────────────────► ⑤ 검색기(Retriever)
     │                                                   (질문과 비슷한 조각 k개)
     │                                                               │
     └──────────────► ⑥ 프롬프트(Prompt) ◄──────────────────────────┘
                      "아래 근거를 보고 질문에 답하세요"
                                │
                                ▼
                      ⑦ LLM(언어모델) ──► 답변 + 출처

   ⑧ 체인(Chain) = ⑤~⑦ 을 한 줄로 이어 붙인 것
```

| 단계 | 하는 일 | 소재 연구 비유 |
|---|---|---|
| 1 문서 로드 | PDF → 텍스트 | 논문을 프린트해 책상에 올려놓기 |
| 2 분할 | 긴 텍스트 → 조각 | 필요한 부분만 포스트잇 크기로 오려 두기 |
| 3 임베딩 | 조각 → 숫자 벡터 | 각 조각에 "내용 지문"(XRD 패턴 같은 특징값) 붙이기 |
| 4 저장 | 벡터 + 원문 보관 | 시료 보관함에 라벨 붙여 정리 |
| 5 검색 | 질문 벡터와 가까운 조각 찾기 | 측정한 XRD 패턴을 데이터베이스 피크와 매칭 |
| 6 프롬프트 | 질문 + 근거를 하나의 지시문으로 | 지도교수님께 "이 자료 보고 답해 주세요" 메모 |
| 7 LLM | 지시문 → 답변 | 연구원이 자료를 읽고 답 작성 |
| 8 체인 | 5~7 자동 연결 | 위 과정을 표준 실험 절차(SOP)로 만들기 |

# Part A. 나이브 구현 — LangChain 체인 없이 직접 만들기

먼저 라이브러리의 "자동 기능" 없이, **numpy 배열** 과 **파이썬 함수** 만으로 8단계를 하나씩 만들어 봅니다.
(PDF 읽기·조각 나누기·임베딩 모델 호출만 도구를 빌려 쓰고, 저장·검색·프롬프트·연결은 전부 직접 합니다.)

## 단계 1: 문서 로드 (Load Documents)

`load_papers()` 는 `paper_data/` 의 PDF 를 **페이지 단위** 로 읽어 `Document` 목록으로 돌려줍니다.
`Document` 는 `page_content`(본문 텍스트) 와 `metadata`(파일 경로, 쪽 번호 등 부가정보) 두 칸을 가진 상자입니다.

In [ ]:
# 단계 1: 문서 로드(Load Documents)
docs = load_papers()

print("읽어 온 페이지 수:", len(docs))
for name in sorted({Path(d.metadata["source"]).name for d in docs}):
    n_pages = sum(Path(d.metadata["source"]).name == name for d in docs)
    print(f" - {name}: {n_pages}쪽")

### (덤) 머리글·바닥글 청소하기

리뷰 논문 PDF 는 **모든 페이지** 위아래에 이런 문구가 반복해서 찍혀 있습니다.

```
This journal is © The Royal Society of Chemistry 2026
J. Mater. Chem. C, 2026, 14, 6558–6580 | 6559
Journal of Materials Chemistry C
Review
```

내용과 아무 관계가 없는 문장이 모든 조각에 섞이면, 임베딩(내용 지문)이 흐려져 검색 정확도가 떨어집니다.
XRD 측정 전에 **기판 피크를 빼 주는 배경 보정(background subtraction)** 과 같은 작업이라고 생각하면 됩니다.

`skku_rag/vector_store.py` 의 `clean_text()` 는 이런 줄만 정규식(문자 패턴)으로 찾아서 지웁니다. 직접 비교해 봅시다.

In [ ]:
from skku_rag.vector_store import clean_text, clean_documents

page = docs[6]  # 리뷰 논문의 2번째 페이지
before = page.page_content.splitlines()
after = clean_text(page.page_content).splitlines()

print("지워진 줄:", [line for line in before if line not in after])
print(f"줄 수: {len(before)} → {len(after)}")

docs = clean_documents(docs)  # 모든 페이지를 청소한 새 목록으로 교체

## 단계 2: 문서 분할 (Split Documents)

한 페이지는 수천 글자라서 통째로 쓰면 ① 검색이 둔해지고 ② LLM 에게 넘길 수 있는 분량을 금방 넘칩니다.
그래서 약 1000자 단위의 **조각(Chunk)** 으로 자릅니다. 조각끼리 100자씩 겹치게(overlap) 해서 문장이 중간에 끊겨도 앞뒤 맥락을 잃지 않게 합니다.

In [ ]:
# 단계 2: 문서 분할(Split Documents)
chunks = split_documents(docs, chunk_size=1000, chunk_overlap=100)

print("조각 수:", len(chunks))
print("조각 하나의 메타데이터(쪽 번호는 0부터 셉니다):", chunks[20].metadata["page"])
print("-" * 60)
print(chunks[20].page_content[:400])

## 단계 3: 임베딩 (Embedding)

각 조각을 임베딩 모델에 넣어 **숫자 벡터(내용 지문)** 로 바꿉니다.
뜻이 비슷한 문장은 벡터도 서로 가까운 방향을 가리킵니다. (조각 180여 개라 수십 초 걸릴 수 있습니다.)

In [ ]:
# 단계 3: 임베딩(Embedding)
embeddings = get_embeddings()

texts = [c.page_content for c in chunks]                 # 조각의 본문만 모은 목록
vectors = np.array(embeddings.embed_documents(texts))   # (조각 수, 벡터 차원) 모양의 숫자 표

print("벡터 표 모양:", vectors.shape)
print("첫 조각 벡터의 앞 5개 숫자:", vectors[0][:5].round(4))

## 단계 4: 벡터DB 저장 (Store)

거창한 데이터베이스가 아니어도 됩니다. **"벡터 표 + 원문 목록 + 출처 목록"** 을 같은 순서로 들고 있으면 그것이 곧 벡터DB 입니다.
`i` 번째 벡터 ↔ `i` 번째 원문 ↔ `i` 번째 출처, 이렇게 번호(index)로 연결됩니다. 시료 보관함의 칸 번호와 라벨 같은 관계입니다.

파일로 저장해 두면 다음에 임베딩을 다시 계산하지 않아도 됩니다.

In [ ]:
# 단계 4: 벡터DB 저장(Store) — numpy 배열 + 파이썬 목록이 곧 '나이브 벡터DB'
sources = [(Path(c.metadata["source"]).name, c.metadata["page"] + 1) for c in chunks]  # (파일명, 사람이 읽는 쪽 번호)

from skku_rag import DB_DIR
np.save(DB_DIR / "naive_vectors.npy", vectors)  # 벡터 표를 파일로 저장
loaded = np.load(DB_DIR / "naive_vectors.npy")  # 다시 불러오기

print("저장 후 다시 읽은 표 모양:", loaded.shape)
print("0번 칸 출처:", sources[0])

## 단계 5: 검색기 (Retriever) — 코사인 유사도 top-k

질문도 같은 임베딩 모델로 벡터로 바꾼 뒤, 저장된 모든 조각 벡터와의 **코사인 유사도(cosine similarity)** 를 계산합니다.

$$\cos(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\lVert\mathbf{q}\rVert\,\lVert\mathbf{d}\rVert}$$

값이 1에 가까울수록 같은 방향(비슷한 내용)입니다. 점수가 높은 순서로 **k 개** 를 뽑으면 검색 끝입니다.
측정한 XRD 패턴을 데이터베이스의 모든 기준 패턴과 비교해 **가장 잘 맞는 상(phase) 몇 개** 를 고르는 것과 같습니다.

In [ ]:
# 단계 5: 검색기(Retriever) — 코사인 유사도로 가장 비슷한 조각 k개 찾기
def search(question: str, k: int = 4) -> list[tuple[float, int]]:
    q = np.array(embeddings.embed_query(question))                             # 질문 → 벡터
    scores = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))  # 모든 조각과 코사인 유사도
    top = np.argsort(scores)[::-1][:k]                                         # 점수 높은 순 k개의 번호
    return [(float(scores[i]), int(i)) for i in top]

question = "구리 배선이 얇아지면 비저항이 커지는 이유는 무엇인가요?"
for score, i in search(question):
    print(f"유사도 {score:.3f} | {sources[i][0][:40]} p.{sources[i][1]} | {texts[i][:60]!r}")

한국어 질문인데도 **영어 논문** 조각이 상위에 나왔는지 확인해 보세요(구체적인 순위와 점수는 임베딩 모델에 따라 다릅니다). 다국어 임베딩 모델은 "언어"가 아니라 "의미"를 벡터로 옮기기 때문에 대개 잘 찾아집니다.

## 단계 6: 프롬프트 (Prompt)

검색된 조각들을 **근거(context)** 로 묶어 질문과 함께 LLM 에게 줄 지시문을 만듭니다. 그냥 파이썬 f-string 이면 충분합니다.
"모르면 모른다고 답하라" 는 지시가 환각을 줄이는 핵심입니다.

In [ ]:
# 단계 6: 프롬프트(Prompt) — f-string 으로 지시문 만들기
def make_prompt(question: str, hits: list[tuple[float, int]]) -> str:
    context = "\n\n".join(f"[출처: {sources[i][0]}, p.{sources[i][1]}]\n{texts[i]}" for _, i in hits)
    return f"""당신은 재료공학 논문을 읽고 답하는 연구 조교입니다.
아래 [근거]만 사용해서 [질문]에 한국어로 답하세요. 근거에 없는 내용이면 "논문에서 찾을 수 없습니다"라고 답하세요.
답변 끝에 참고한 출처(파일명, 쪽)를 적으세요.

[근거]
{context}

[질문]
{question}"""

print(make_prompt(question, search(question, k=2))[:900])

## 단계 7: 언어모델 (LLM)

완성된 프롬프트를 LLM 에 넣고 답을 받습니다. `get_llm()` 은 Ollama 로 실행 중인 로컬 모델을 돌려줍니다.

In [ ]:
# 단계 7: 언어모델(LLM)
llm = get_llm()

response = llm.invoke(make_prompt(question, search(question)))
print(response.content)

## 단계 8: 체인 (Chain) — `naive_rag()` 함수 하나로 묶기

단계 5~7 을 하나의 함수로 이어 붙이면 그것이 **체인** 입니다. 출처는 LLM 에게만 맡기지 않고 **코드가 직접** 찍어 줍니다(LLM 이 출처를 빼먹거나 지어낼 수 있기 때문).

In [ ]:
# 단계 8: 체인(Chain) — 검색 → 프롬프트 → LLM 을 한 함수로
def naive_rag(question: str, k: int = 4) -> str:
    hits = search(question, k)                       # 5. 검색
    prompt = make_prompt(question, hits)             # 6. 프롬프트
    answer = get_llm().invoke(prompt).content        # 7. LLM
    refs = sorted({f"{sources[i][0]} p.{sources[i][1]}" for _, i in hits})
    return answer + "\n\n[검색된 출처]\n" + "\n".join(" - " + r for r in refs)

print(naive_rag("Cu 합금에 Nb를 첨가하면 결정립 크기와 비저항은 어떻게 변하나요?"))

**여기까지가 RAG 의 전부입니다.** numpy 배열 하나, 함수 세 개(`search`, `make_prompt`, `naive_rag`)로 논문 기반 질의응답이 됩니다.
이제 같은 것을 LangChain 으로 만들어 보고, 무엇이 편해지는지 비교합니다.

# Part B. 같은 것을 LangChain 으로

| Part A (직접 구현) | Part B (LangChain) |
|---|---|
| `load_papers()` + `clean_documents()` + `split_documents()` | (동일) |
| `np.array(embeddings.embed_documents(...))` + `np.save` | `Chroma` 벡터DB (`get_vectorstore()`) |
| `search(question, k)` | `vectorstore.as_retriever(search_kwargs={"k": 4})` |
| `make_prompt()` f-string | `PromptTemplate` |
| `get_llm().invoke(prompt).content` | `llm` + `StrOutputParser()` |
| `naive_rag()` 함수 | LCEL 체인 `{...} | prompt | llm | parser` |

## 단계 1~4: 사전작업 — `get_vectorstore()`

`skku_rag/vector_store.py` 의 `get_vectorstore()` 는 Part A 의 단계 1~4(로드 → 청소 → 분할 → 임베딩 → 저장)를 한 번에 수행합니다.
저장 장소가 numpy 파일 대신 **Chroma**(03장에서 배운 벡터DB) 이고, `db/chroma` 폴더에 남기 때문에 **두 번째부터는 바로 불러옵니다**.

In [ ]:
# 단계 1~4: 문서 로드 → 분할 → 임베딩 → 벡터DB 저장 (이미 있으면 불러오기만)
from skku_rag.vector_store import get_vectorstore, format_docs

vectorstore = get_vectorstore()  # 처음부터 다시 만들려면 get_vectorstore(rebuild=True)
print("Chroma 에 저장된 조각 수:", vectorstore._collection.count())

## 단계 5: 검색기 (Retriever)

`as_retriever()` 는 벡터DB 를 "질문을 넣으면 조각 k 개를 돌려주는 부품" 으로 바꿔 줍니다. Part A 의 `search()` 와 같은 역할입니다.

In [ ]:
# 단계 5: 검색기(Retriever) 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})  # 가장 비슷한 조각 4개

found = retriever.invoke(question)
for d in found:
    print(Path(d.metadata["source"]).name[:40], "p.", d.metadata["page"] + 1, "|", d.page_content[:60].replace("\n", " "))

Part A 의 `search()` 결과와 거의 같은 조각이 나옵니다(같은 임베딩, 같은 조각이니까요).

검색된 `Document` 목록을 프롬프트에 넣으려면 하나의 문자열로 합쳐야 합니다. 그 일을 하는 `format_docs()` 도 `vector_store.py` 에 넣어 두었습니다(08장에서 재사용).

In [ ]:
print(format_docs(found[:2])[:700])

## 단계 6: 프롬프트 (Prompt)

`PromptTemplate` 은 `{context}`, `{question}` 이라는 **빈칸이 있는 양식** 입니다. Part A 의 f-string 과 같지만, 체인에 끼울 수 있는 부품이 됩니다.

In [ ]:
# 단계 6: 프롬프트(Prompt) 생성
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 재료공학 논문을 읽고 답하는 연구 조교입니다.
아래 [근거]만 사용해서 [질문]에 한국어로 답하세요.
근거에서 답을 찾을 수 없으면 지어내지 말고 "모르겠습니다(논문에서 찾을 수 없습니다)"라고 답하세요.
답변 끝에 참고한 출처를 (파일명, p.쪽) 형식으로 적으세요.

[근거]
{context}

[질문]
{question}

[답변]"""
)
print(prompt.input_variables)  # 채워야 할 빈칸 이름

## 단계 7: 언어모델 (LLM)

In [ ]:
# 단계 7: 언어모델(LLM) 생성
llm = get_llm()

## 단계 8: 체인 (Chain) — LCEL 로 연결

LangChain 의 **LCEL(LangChain Expression Language)** 은 `|` (파이프) 기호로 부품을 이어 붙입니다. 왼쪽의 출력이 오른쪽의 입력으로 흘러갑니다.

```
질문 ─┬─► retriever ─► format_docs ─► context ─┐
      └─► RunnablePassthrough(그대로 통과) ─► question ─┴─► prompt ─► llm ─► StrOutputParser ─► 답변 문자열
```

- `{"context": ..., "question": ...}` : 질문 하나를 받아 두 갈래로 나눠 빈칸을 채웁니다.
- `RunnablePassthrough()` : 받은 것을 아무것도 안 하고 그대로 넘깁니다(질문 원문).
- `StrOutputParser()` : LLM 의 응답 메시지에서 **글자만** 꺼냅니다(`.content` 와 같은 일).

In [ ]:
# 단계 8: 체인(Chain) 생성
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("루테늄(Ru)이 아주 얇은 배선에서 구리보다 유리할 수 있는 이유는 무엇인가요?"))

### 스트리밍(Streaming) 출력

`invoke()` 는 답이 다 만들어질 때까지 기다렸다가 한 번에 보여 줍니다. `stream()` 을 쓰면 ChatGPT 처럼 **글자가 생기는 대로** 바로 출력됩니다.

In [ ]:
for token in chain.stream("나노결정 Cu 합금 연구에서 Fe 첨가는 어떤 결과를 보였나요?"):
    print(token, end="", flush=True)  # 조각(token)이 도착할 때마다 바로 출력

Part A 의 약 40줄이 Part B 에서는 체인 몇 줄로 줄었습니다. 하지만 **안에서 일어나는 일은 똑같습니다** — 벡터 검색 → 문자열 합치기 → 프롬프트 채우기 → LLM 호출.
라이브러리는 이것을 **교체 가능한 부품** 으로 포장해 준 것뿐이라, 예를 들어 Chroma 를 다른 벡터DB 로, Ollama 를 다른 LLM 으로 바꿔도 체인 모양은 그대로입니다.

# Part C. 실험 — 벡터 RAG 를 시험해 보기

## C-1. 여러 질문 던져 보기

두 논문 각각에 대한 질문과, **두 논문을 아우르는** 질문을 해 봅니다. 답과 함께 "어느 조각을 근거로 썼는지" 도 확인합니다.

In [ ]:
def ask(question: str, k: int = 4) -> None:
    """질문 → 검색된 조각 출처 출력 → 체인 답변 출력"""
    print("■ 질문:", question)
    for d in vectorstore.as_retriever(search_kwargs={"k": k}).invoke(question):
        print(f"   검색: {Path(d.metadata['source']).name[:35]:35s} p.{d.metadata['page'] + 1}")
    print("■ 답변:", chain.invoke(question), "\n" + "=" * 70)

In [ ]:
ask("Kim et al. 논문에서 Cu에 Nb 또는 Fe를 소량 첨가했을 때, 비저항이 순수 나노결정 Cu보다 낮아진 이유는 무엇인가요?")

In [ ]:
ask("구리를 대체할 배선 금속을 고를 때 전자 평균자유행로(EMFP)가 왜 중요한가요?")

In [ ]:
ask("두 논문 모두에서 구리의 비저항을 높이는 요인으로 언급되는 것은 무엇인가요?")

**관찰 포인트**

- 한 논문 안에 답이 모여 있는 질문(1, 2번)은 대개 잘 답합니다. 답과 출처가 맞는지 확인해 보세요.
- 두 논문을 아우르는 질문(3번)은 검색된 4개 조각이 **한 논문에 쏠리면** 다른 논문 내용이 답에서 빠집니다. 출처 목록에 두 파일이 모두 나왔는지 확인해 보세요.
- 4B 소형 모델이라 출처 표기 형식을 지키지 않거나 근거를 약간 과장할 수 있습니다. 그래서 Part A 에서는 출처를 코드로 직접 찍었습니다.

## C-2. 조각 크기(chunk_size)와 검색 개수(k) 바꿔 보기

- **chunk_size** : 조각이 작으면 검색이 "정밀"해지지만 앞뒤 맥락이 잘립니다. 크면 맥락은 풍부하지만 엉뚱한 내용이 섞입니다.
- **k** : 많이 가져오면 놓치는 정보는 줄지만, LLM 이 읽을 분량이 늘고 잡음도 늘어납니다.

시료를 너무 잘게 자르면 미세조직(맥락)이 안 보이고, 너무 크게 자르면 원하는 상(phase)을 찾기 어려운 것과 비슷합니다.

비교를 위해 chunk_size=300 짜리 벡터DB 를 **메모리에만** 임시로 만들어 봅니다(`db/` 에 저장하지 않음).

In [ ]:
from langchain_chroma import Chroma
from skku_rag.vector_store import load_chunks

small_chunks = load_chunks(chunk_size=300, chunk_overlap=30)  # 작은 조각
small_ids = [f"c300_{i:04d}" for i in range(len(small_chunks))]   # 고정 id → 다시 실행해도 중복 저장 안 됨
small_store = Chroma.from_documents(small_chunks, get_embeddings(), ids=small_ids,
                                    collection_name="exp_chunk300")

print("chunk_size=1000 조각 수:", vectorstore._collection.count())
print("chunk_size=300  조각 수:", len(small_chunks))

In [ ]:
q = "Cu와 Ru의 상온 비저항과 EMFP 값은 각각 얼마인가요?"
for name, store in [("chunk 1000", vectorstore), ("chunk 300", small_store)]:
    top = store.similarity_search(q, k=2)
    print(f"[{name}] 1위 조각 ({len(top[0].page_content)}자, p.{top[0].metadata['page'] + 1}):")
    print("   ", top[0].page_content[:250].replace("\n", " "), "\n")

같은 질문이라도 조각 크기에 따라 1위로 뽑히는 조각과 그 안에 담긴 정보량이 달라질 수 있습니다. 두 결과를 비교해 보세요.
특히 **표(Table)** 는 PDF 에서 숫자가 한 줄에 하나씩 세로로 풀려 나오기 때문에, 작게 자르면 "Cu" 와 "1.678" 이 서로 다른 조각으로 떨어져 버리기도 합니다.

이번에는 **k** 를 바꿔 답이 어떻게 달라지는지 봅니다.

In [ ]:
for k in [1, 6]:
    k_chain = (
        {"context": vectorstore.as_retriever(search_kwargs={"k": k}) | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    print(f"===== k={k} =====")
    print(k_chain.invoke(q), "\n")

정답은 리뷰 논문 Table 1 의 **Cu: 1.678 μΩ·cm, 39.9 nm / Ru: 7.8 μΩ·cm, 6.59(또는 4.88) nm** 입니다. 모델의 답을 직접 대조해 보세요.

- k=1 처럼 근거가 너무 적으면 "찾을 수 없다" 고 하거나 일부만 답하기 쉽습니다.
- k 를 늘리면 정답 조각이 들어올 확률은 올라가지만, **정작 Table 1 조각이 검색되지 않으면** 모델이 "표 1 참조" 같은 말을 붙이며 **기억 속 값을 지어내기도** 합니다(숫자·단위를 꼭 대조하세요).
- Table 1 조각은 검색에서 잘 안 잡히는 경우가 많습니다. 왜 그럴까요? 그 이유는 C-4 에서 표 조각을 직접 열어 보며 확인합니다.

## C-3. 논문에 없는 질문을 하면?

두 논문에는 그래핀 이야기가 없습니다. 이때 RAG 가 "모른다" 고 답할 수 있는지 봅시다.
먼저 Part A 의 `search()` 로 **유사도 점수** 를 비교해 보면, 논문에 없는 질문은 대개 1위 조각의 점수부터 낮습니다(점수의 절대값은 임베딩 모델마다 다릅니다).

In [ ]:
for q in ["구리 배선이 얇아지면 비저항이 커지는 이유는 무엇인가요?", "그래핀의 밴드갭은 얼마인가요?"]:
    best_score, _ = search(q, k=1)[0]
    print(f"1위 유사도 {best_score:.3f}  ← {q}")

In [ ]:
graphene_q = "그래핀의 밴드갭은 얼마인가요?"
print("[LLM 단독]\n", llm.invoke(graphene_q).content[:400], "\n")
print("[RAG]\n", chain.invoke(graphene_q))

- **LLM 단독** 은 자기 기억(사전학습 지식)으로 답합니다. 이번 질문은 상식이라 맞을 수도 있지만, 근거·출처가 없고 틀려도 알 길이 없습니다.
- **RAG** 는 프롬프트에서 "근거에 없으면 모른다고 답하라" 고 했기 때문에 "찾을 수 없습니다" 쪽으로 답하도록 유도됩니다.
  소형 모델은 이 지시를 어기고 자기 지식으로 답을 덧붙이기도 합니다 — 이런 경우를 막으려면 **유사도 점수가 일정 값 이하면 아예 LLM 을 부르지 않는** 장치를 둘 수 있습니다(직접 해보기 2번).

## C-4. 벡터 RAG 의 한계

벡터 RAG 는 "질문과 **문장이 비슷한** 조각 몇 개" 를 가져옵니다. 그래서 다음 같은 질문에 약합니다.

1. **여러 조각에 흩어진 관계** : "A 는 B 에 영향을 주고, B 는 C 와 관련 있다" 처럼 사실들이 서로 다른 페이지에 있을 때
2. **수치 비교·계산** : 표의 여러 행을 모아 정렬·비교·계산해야 할 때 (LLM 은 계산에 약하고, 표는 조각으로 잘려 있음)

In [ ]:
hard_q = "리뷰 논문 Table 1의 금속들(Cu, Al, Co, Ru, Mo, Ir)을 EMFP가 짧은 순서대로 정렬하고, 각 값을 적어 주세요."
ask(hard_q, k=4)

정답(Table 1, 상온 기준, 두 값이 있는 hcp 금속은 큰 값 기준): **Ru 6.59 < Ir 7.09 < Mo 11.2 < Co 11.8 < Al 18.9 < Cu 39.9 nm**.

결과를 대조해 보면 "찾을 수 없다" 고 하거나, 값이 빠지거나, 숫자를 엉뚱한 금속에 붙이는 일이 생기기 쉽습니다. Table 1 이 들어 있는 조각을 직접 열어 봅시다.

In [ ]:
# Cu 의 비저항 값 "1.678" 이 들어 있는 조각 찾기
table_chunk = next(c for c in chunks if "1.678" in c.page_content)
lines = table_chunk.page_content.splitlines()
# 표 머리글('Element' 가 들어 있는 첫 줄)부터. 없으면 조각 처음부터 보여 줌
start = next((i for i, line in enumerate(lines) if "Element" in line), None)
if start is None:
    print("(머리글 'Element' 줄을 찾지 못해 조각 처음부터 보여 줍니다)")
    start = 0
print(" | ".join(lines[start:start + 30]))  # 줄바꿈 대신 | 로 이어서 출력

사람 눈에는 표였던 것이 조각 안에서는 `Cu43 | fcc | 1.678 | 39.9 | 6.70 | Al43 | ...` 처럼 **한 줄에 값 하나씩** 풀려 있습니다.
게다가 PDF 글꼴 문제로 `μΩ cm` 은 `mO cm`, `λ` 는 `l` 로 깨졌고, 참고문헌 번호 43 이 원소기호에 붙어 `Cu43` 이 되었습니다.
이런 조각은 "EMFP 가 짧은 금속" 같은 **문장형 질문과 임베딩이 닮지 않아** 검색 순위에서 밀리기 쉽습니다.

정리하면 벡터 RAG 가 이 질문에 약한 이유는
- 표가 PDF 에서 **숫자 한 줄씩 세로로 풀려** 조각에 들어가 있어 "어느 숫자가 어느 금속 것인지" 구조가 깨졌고,
- 검색은 "비슷한 문장" 을 찾을 뿐 **표 전체를 빠짐없이** 모아 오지 않으며,
- 정렬·비교는 LLM 이 머릿속으로 해야 하기 때문입니다.

이 한계를 넘는 두 가지 방향을 다음 장들에서 배웁니다.

| 문제 | 해결 방향 | 장 |
|---|---|---|
| 흩어진 **관계** 를 이어서 따라가야 함 | 논문에서 (개체)–[관계]→(개체) 를 뽑아 **그래프** 로 저장하고 그래프를 따라 검색 | **07 Graph RAG** |
| 여러 번 검색하고, 표를 조회하고, **계산** 까지 해야 함 | 검색을 하나의 **도구(Tool)** 로 주고, LLM 이 스스로 "검색 → 확인 → 다시 검색/계산" 을 반복 | **08 Agentic RAG** (05 에이전트 기초 필요) |

## 정리

- RAG = **사전작업**(① 로드 → ② 분할 → ③ 임베딩 → ④ 저장) + **실행**(⑤ 검색 → ⑥ 프롬프트 → ⑦ LLM), 이를 이어 붙인 것이 ⑧ 체인입니다.
- Part A 에서 numpy 배열 + 코사인 유사도 + f-string 만으로 `naive_rag()` 를 완성했습니다. **벡터DB 의 본질은 "벡터 표 + 원문 + 출처" 와 "가장 가까운 것 찾기"** 입니다.
- Part B 의 LangChain 버전(`get_vectorstore` → `as_retriever` → `PromptTemplate` → LCEL 체인)은 같은 일을 교체 가능한 부품으로 포장한 것입니다.
- 머리글/바닥글 같은 반복 문구는 미리 청소(`clean_text`)하면 검색 품질에 도움이 됩니다.
- `chunk_size`, `k` 는 정답이 없는 조절 손잡이입니다. 질문 유형과 문서 형태(본문 vs 표)에 맞춰 실험으로 정합니다.
- 프롬프트의 "모르면 모른다고 답하라" 는 환각을 줄이지만 소형 모델은 어길 수 있으므로, 유사도 기준이나 출처 표시 같은 **코드 차원의 안전장치** 가 함께 필요합니다.
- 벡터 RAG 는 **흩어진 관계** 와 **수치 비교·계산** 에 약합니다 → 07 Graph RAG, 08 Agentic RAG.

## 직접 해보기

1. **나만의 질문** : 두 논문에서 궁금한 점(예: "CuAl2 같은 금속간화합물이 배선 후보로 주목받는 이유")을 `naive_rag()` 와 `chain.invoke()` 에 각각 넣고, 검색된 출처와 답이 같은지 비교해 보세요.
2. **안전장치 추가** : `naive_rag()` 를 고쳐서, 1위 조각의 유사도가 기준값(예: C-3 에서 본 두 점수의 중간값)보다 낮으면 LLM 을 부르지 않고 `"논문에서 관련 내용을 찾지 못했습니다."` 를 돌려주게 만들어 보세요.
3. **조절 손잡이 실험** : `load_chunks(chunk_size=..., chunk_overlap=...)` 로 chunk_size 를 500, 2000 으로 바꾼 임시 Chroma 를 만들고, C-4 의 표 질문 정답률이 어떻게 변하는지 기록해 보세요. (힌트: 표가 한 조각에 통째로 들어가면 유리할까요?)